# 02 - Training and submission pipeline

This notebook runs the same end-to-end pipeline used for the submitted experiments:
it installs the dependencies, loads the competition data, fine-tunes an encoder-decoder
model (AfriTeVa-V2 / mT5), and writes a Zindi submission file. It runs top-to-bottom on
Google Colab (T4/L4) or a local CUDA GPU.

**Colab quick start:** open this notebook in Colab (badge below), set Runtime to a GPU,
then Run all. The only manual step is uploading the four competition CSVs when prompted
(they are CC-BY-SA and not committed to the repo).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jbyiringiro/multilingual-health-qa/blob/main/notebooks/02_Train.ipynb)


## 1. Setup
The cell below installs the project dependencies. On Colab it first clones the
repository and changes into it automatically (detected via `google.colab`); when
running locally those two steps are skipped because the repo is already present.

**If the runtime disconnects:** Colab resets the kernel and working directory, so
run `Runtime -> Run all` again (this cell is idempotent and re-enters the repo).
Training restarts from the beginning - for a quick check use the smoke-test line in
the training cell so a full run finishes before any disconnect.


In [ ]:
import os, sys

# On Colab, make sure we are inside the repo (idempotent across reconnects):
#   - already inside it  -> 'src' exists in cwd, do nothing
#   - cloned but cwd reset-> the repo folder exists, just cd into it
#   - fresh VM           -> clone, then cd in
REPO = 'multilingual-health-qa'
if 'google.colab' in sys.modules and not os.path.isdir('src'):
    if not os.path.isdir(REPO):
        os.system(f'git clone -q https://github.com/jbyiringiro/{REPO}.git')
    os.chdir(REPO)

os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
!pip -q install -r requirements.txt


## 2. Data
The competition data (CC-BY-SA) is not committed to the repository. On Colab the cell
below prompts you to upload `Train.csv`, `Val.csv`, `Test.csv` and `SampleSubmission.csv`
(or mount Google Drive instead - that line is commented). It then verifies the required
CSVs are present in `data/` before training.


In [ ]:
import os, sys
os.makedirs('data', exist_ok=True)

# The competition CSVs (CC-BY-SA) are not committed. Provide them once:
#   download Train.csv, Val.csv, Test.csv, SampleSubmission.csv from the Zindi page.
NEED = ['Train.csv', 'Val.csv', 'Test.csv']

if 'google.colab' in sys.modules and not all(os.path.exists(f'data/{n}') for n in NEED):
    from google.colab import files
    print('Upload Train.csv, Val.csv, Test.csv, SampleSubmission.csv:')
    uploaded = files.upload()
    for name in uploaded:
        os.replace(name, f'data/{name}')
    # Alternative to uploading - mount Drive and copy:
    # from google.colab import drive; drive.mount('/content/drive')
    # !cp '/content/drive/MyDrive/healthqa/'*.csv data/

present = [f for f in os.listdir('data') if f.endswith('.csv')]
print('data files:', present)
missing = [n for n in NEED if not os.path.exists(f'data/{n}')]
assert not missing, f'Missing in data/: {missing} - add them before training.'


## 3. Training
Each YAML in `configs/` defines one reproducible experiment. The cell below loads a
configuration and calls `train()`, which fine-tunes the model, evaluates it on the
validation set, appends the metrics to the log, and writes a submission file. The
submitted results used the AfriTeVa-V2 configurations; the small baseline is shown here
as the simplest example, and the commented line enables a fast smoke test.

In [ ]:
from src.config import Config
from src.train import train
cfg = Config.from_yaml('configs/mt5_small_baseline.yaml')
# Smoke test (uncomment): runs the full loop on a tiny subsample in ~2 min
# cfg.max_train_samples = 200; cfg.max_eval_samples = 100; cfg.epochs = 1
metrics = train(cfg)            # fine-tunes, evaluates, logs, and writes a submission
metrics

## 4. The generated submission
The submission has the three independently-scored target columns (`TargetRLF1`,
`TargetR1F1`, `TargetLLM`). The cell below loads the file `train()` produced and
previews its first rows.

In [ ]:
import pandas as pd
sub = pd.read_csv(cfg.submission_path)
print(sub.shape); sub.head(3)

## 5. Inference from a saved model
`generate_submission` rebuilds a submission from an already-trained model without
retraining - for example to re-decode an existing checkpoint with different settings.

In [ ]:
# from src.predict import generate_submission
# generate_submission(cfg)       # optionally pass model_dir=... / out=...

## 6. Experiment log
Each call to `train()` appends a row to `experiments/results.csv`, the single source
of truth behind the report's experiment table and leaderboard-progression figure.

In [ ]:
import pandas as pd
pd.read_csv('experiments/results.csv')